In [ ]:
import os
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

In [ ]:
# ---------------- Paths ----------------
catalog_file = r"directory\CNN_Catalog_2024.csv"
signal_directory = r"directory\Merged_Signals_2018_2025"


In [ ]:
# ---------------- Time adjustment to make sure the detected start and peak times are in local extrema of the signal ----------------
def adjust_event_times(signal_df, start_time, peak_time):
    """
    Find local min near start_time and local max near peak_time.
    """
    bt = pd.to_datetime(signal_df["time"])
    flux = signal_df["Long_Channel_Flux"].values

    # find closest indices
    idx_start = (np.abs(bt - start_time)).argmin()
    idx_peak = (np.abs(bt - peak_time)).argmin()

    # search window (±1 minutes around CNN time)
    window = pd.Timedelta(minutes=1)

    # Local min around CNN start
    mask_s = (bt >= start_time - window) & (bt <= start_time + window)
    if mask_s.any():
        idx_local_min = flux[mask_s].argmin() + np.where(mask_s)[0][0]
    else:
        idx_local_min = idx_start

    # Local max around CNN peak
    mask_p = (bt >= peak_time - window) & (bt <= peak_time + window)
    if mask_p.any():
        idx_local_max = flux[mask_p].argmax() + np.where(mask_p)[0][0]
    else:
        idx_local_max = idx_peak

    # Enforce ordering: start must be before peak
    if bt[idx_local_min] >= bt[idx_local_max]:
        # fallback to CNN times
        idx_local_min = min(idx_start, idx_peak)
        idx_local_max = max(idx_start, idx_peak)

    return bt[idx_local_min], flux[idx_local_min], bt[idx_local_max], flux[idx_local_max]

# ---------------- Main ----------------
# Read catalog
catalog_df = pd.read_csv(catalog_file, parse_dates=["date", "event_start", "event_peak"])

adjusted_rows = []

# Process day by day
for d in catalog_df["date"].dt.date.unique():
    print(d)
    date_str_ = datetime.strftime(d, "%Y%m%d")
    fname = f"GOES_Signal_{date_str_}.csv"
    fpath = os.path.join(signal_directory, fname)

    if not os.path.exists(fpath):
        print(f"⚠️ Missing file for {d}: {fname}")
        continue

    signal_df = pd.read_csv(fpath, parse_dates=["time"])

    day_events = catalog_df[catalog_df["date"].dt.date == d].copy()

    for idx, row in day_events.iterrows():
        adj_start, start_val, adj_peak, peak_val = adjust_event_times(
            signal_df, row["event_start"], row["event_peak"]
        )
        # Update row
        row["event_start"] = adj_start
        row["start_value"] = start_val
        row["event_peak"] = adj_peak
        row["peak_value"] = peak_val

        adjusted_rows.append(row)

# Combine all adjusted rows
adjusted_df = pd.DataFrame(adjusted_rows)

# ---------------- Enforce monotonic peaks across events ----------------
adjusted_df = adjusted_df.sort_values("event_start").reset_index(drop=True)

for i in range(1, len(adjusted_df)):
    if adjusted_df.loc[i, "event_peak"] <= adjusted_df.loc[i-1, "event_peak"]:
        # fallback to CNN catalog peak
        original_peak = catalog_df.loc[adjusted_df.loc[i].name, "event_peak"]

        if original_peak > adjusted_df.loc[i-1, "event_peak"]:
            adjusted_df.loc[i, "event_peak"] = original_peak
        else:
            # minimal shift forward
            adjusted_df.loc[i, "event_peak"] = (
                adjusted_df.loc[i-1, "event_peak"] + pd.Timedelta(seconds=1)
            )

# ---------------- Excluding one point events! ----------------
adjusted_df2 = adjusted_df[adjusted_df["event_peak"] != adjusted_df["event_start"]]


In [ ]:
# # Merging the overlaping and very close events 
df = adjusted_df2.sort_values("event_start").reset_index(drop=True)
threshold = pd.Timedelta(seconds=60)

merged = []

# start new group
current_group = [df.loc[0]]
current_start = df.loc[0, "event_start"]
current_end   = df.loc[0, "event_peak"]

for i in range(1, len(df)):
    row = df.loc[i]

    # compare against current group’s latest end, not just prev row
    if row["event_start"] <= current_end + threshold:
        current_group.append(row)
        # update the running end of the group
        current_end = max(current_end, row["event_peak"])
    else:
        # finalize current group
        group_df = pd.DataFrame(current_group)

        idx_max_flux = group_df["peak_value"].idxmax()
        
        merged.append({
            "date": group_df["date"].min(),  
            "event_start": group_df["event_start"].min(),   # earliest start
            "start_value": group_df.loc[group_df["event_start"].idxmin(), "start_value"],  # value at earliest start
            "event_peak": group_df.loc[idx_max_flux, "event_peak"],    # peak time of max flux
            "peak_value": group_df.loc[idx_max_flux, "peak_value"],    # max flux
            "merged_from": group_df.index.tolist()
        })

        # start new group
        current_group = [row]
        current_start = row["event_start"]
        current_end   = row["event_peak"]

# finalize last group
group_df = pd.DataFrame(current_group)
idx_max_flux = group_df["peak_value"].idxmax()
merged.append({
    "date": group_df["date"].min(),  
    "event_start": group_df["event_start"].min(),
    "start_value": group_df.loc[group_df["event_start"].idxmin(), "start_value"],
    "event_peak": group_df.loc[idx_max_flux, "event_peak"],
    "peak_value": group_df.loc[idx_max_flux, "peak_value"],
    "merged_from": group_df.index.tolist()
})

merged_df = pd.DataFrame(merged)


In [ ]:
# ---------------- Save ----------------
merged_df.to_csv("Refined_CNN_Catalog_2024.csv", index=False)
print(len(catalog_df), len(adjusted_df))
print("Done!")

In [ ]:
import matplotlib.dates as mdates
import matplotlib.pyplot as plt

date_str = "20240323"
fname = f"GOES_Signal_{date_str}.csv"
fpath = os.path.join(signal_directory, fname)

# --- Load signal ---
signal_df = pd.read_csv(fpath, parse_dates=["time"])
tt_ = signal_df["time"]
S = signal_df["Long_Channel_Flux"]

# --- Extract CNN and adjusted events ---
cnn_events = catalog_df[catalog_df["date"].dt.strftime("%Y%m%d") == date_str]
adj_events = merged_df[merged_df["date"].dt.strftime("%Y%m%d") == date_str]

# Labels dict (background=0, rise=1)
colors = ['blue', 'red']
labels_dict = {0: 'background', 1: 'rise'}

# =========================================================
# --- Plot CNN raw events vs Refined times ---
fig, axes = plt.subplots(2, 1, figsize=(15, 8), sharex=True)

# ---------------- Panel 1: CNN ----------------
ax = axes[0]
labels = np.zeros(len(S), dtype=int)
for _, row in cnn_events.iterrows():
    mask = (tt_ >= row["event_start"]) & (tt_ <= row["event_peak"])
    labels[mask] = 1

for phase in [0, 1]:
    mask = labels == phase
    ax.scatter(tt_[mask], S[mask], s=1.5,
               label=labels_dict[phase] if phase == 1 else None,
               color=colors[phase])

# for pk in pd.to_datetime(cnn_events["event_peak"].unique()):
#     ax.axvline(pk, color='k', linestyle='--', linewidth=0.8)

ax.set_title(f"CNN Catalog Intervals — {date_str}")
ax.set_yscale("log")
ax.legend()
ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
ax.set_ylabel("Flux")

# ---------------- Panel 2: Adjusted ----------------
ax = axes[1]
labels_adj = np.zeros(len(S), dtype=int)
for _, row in adj_events.iterrows():
    mask = (tt_ >= row["event_start"]) & (tt_ <= row["event_peak"])
    labels_adj[mask] = 1

for phase in [0, 1]:
    mask = labels_adj == phase
    ax.scatter(tt_[mask], S[mask], s=1.5,
               label=labels_dict[phase] if phase == 1 else None,
               color=colors[phase])

# for pk in pd.to_datetime(adj_events["event_peak"].unique()):
#     ax.axvline(pk, color='k', linestyle='--', linewidth=0.8)

ax.set_title(f"Time-Adjusted Intervals — {date_str}")
ax.set_yscale("log")
ax.legend()
ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
ax.set_xlabel("Time")
ax.set_ylabel("Flux")

plt.tight_layout()
plt.savefig(r"C:\Users\nfar4944\OneDrive - The University of Sydney (Staff)\Desktop\F1.png",
            dpi=300, bbox_inches="tight")
plt.show()
